# Lesson 8: Logistic Regression

*ML Academy — companion notebook*

Classification with the sigmoid, probabilities, log loss and decision boundaries.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-08.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 11. In Python

In [1]:
import numpy as np

x = np.array([0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5])
y = np.array([0, 0, 0, 1, 0, 1, 0, 1, 1, 1])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

w, b, lr = 0.0, 0.0, 0.1
for step in range(5000):
    p = sigmoid(w * x + b)
    w -= lr * np.mean((p - y) * x)      # dL/dw = mean((p - y) x)
    b -= lr * np.mean(p - y)            # dL/db = mean(p - y)

p = sigmoid(w * x + b)
loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
print(round(w, 3), round(b, 3), round(loss, 3))   # 1.353 -3.721 0.434
print(round(-b / w, 2))                           # decision boundary: 2.75 hours

1.353 -3.721 0.434
2.75


In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

print(round(model.score(X_test, y_test), 3))      # accuracy on unseen tumours
print(model.predict_proba(X_test[:3]).round(3))   # [P(class 0), P(class 1)] for 3 tumours

0.958
[[0.004 0.996]
 [1.    0.   ]
 [1.    0.   ]]


## 14. Going deeper: logistic regression as a probability model

In [3]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True)
X = StandardScaler().fit_transform(X[:, :5])          # 5 features keep the example small
A = np.column_stack([np.ones(len(y)), X])
lam = 1.0                                             # L2 penalty, same as sklearn's C = 1

w = np.zeros(A.shape[1])
for it in range(1, 11):                               # Newton's method (IRLS)
    p = 1 / (1 + np.exp(-A @ w))
    grad = A.T @ (p - y) + lam * np.r_[0, w[1:]]      # don't penalise the bias
    S = p * (1 - p)                                   # each point's weight
    H = A.T @ (A * S[:, None]) + lam * np.diag(np.r_[0, np.ones(5)])
    step = np.linalg.solve(H, grad)
    w -= step
    print(f"iteration {it}: step size {np.linalg.norm(step):.2e}")
    if np.linalg.norm(step) < 1e-8: break

sk = LogisticRegression(C=1.0, tol=1e-10, max_iter=1000).fit(X, y)   # tight tolerance
print("ours:   ", w.round(4))
print("sklearn:", np.r_[sk.intercept_, sk.coef_[0]].round(4))

iteration 1: step size 1.78e+00
iteration 2: step size 8.46e-01
iteration 3: step size 1.09e+00
iteration 4: step size 7.33e-01
iteration 5: step size 3.00e-01
iteration 6: step size 4.06e-02
iteration 7: step size 6.29e-04
iteration 8: step size 1.48e-07
iteration 9: step size 7.89e-15
ours:    [ 0.7707 -1.1797 -1.405  -1.8599 -1.5528 -1.7102]
sklearn: [ 0.7707 -1.1797 -1.405  -1.8599 -1.5528 -1.7102]


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import brier_score_loss

Xa, ya = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(Xa, ya, test_size=0.5, random_state=0, stratify=ya)
sc = StandardScaler().fit(Xtr)
models = {"logistic regression": LogisticRegression(max_iter=1000).fit(sc.transform(Xtr), ytr),
          "naive Bayes": GaussianNB().fit(sc.transform(Xtr), ytr)}
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
for name, m in models.items():
    p = m.predict_proba(sc.transform(Xte))[:, 1]
    print(f"{name}: Brier score {brier_score_loss(yte, p):.3f}")
    idx = np.digitize(p, bins[1:-1])
    for b in range(5):
        sel = idx == b
        if sel.sum(): print(f"   predicted {bins[b]:.1f}–{bins[b+1]:.1f}: {sel.sum():3d} cases, average prediction {p[sel].mean():.2f}, actually positive {yte[sel].mean():.2f}")

logistic regression: Brier score 0.018
   predicted 0.0–0.2:  98 cases, average prediction 0.02, actually positive 0.00
   predicted 0.2–0.4:   6 cases, average prediction 0.32, actually positive 0.33
   predicted 0.4–0.6:   6 cases, average prediction 0.49, actually positive 0.67
   predicted 0.6–0.8:   7 cases, average prediction 0.72, actually positive 1.00
   predicted 0.8–1.0: 168 cases, average prediction 0.98, actually positive 0.99
naive Bayes: Brier score 0.071
   predicted 0.0–0.2: 107 cases, average prediction 0.00, actually positive 0.10
   predicted 0.6–0.8:   1 cases, average prediction 0.69, actually positive 1.00
   predicted 0.8–1.0: 177 cases, average prediction 1.00, actually positive 0.94


In [5]:
from sklearn.metrics import recall_score, precision_score

# Imbalanced problem: 5% positives
from sklearn.datasets import make_classification
Xi, yi = make_classification(n_samples=4000, n_features=8, weights=[0.95], class_sep=0.8, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(Xi, yi, test_size=0.5, random_state=0, stratify=yi)
for cw in [None, "balanced"]:
    m = LogisticRegression(class_weight=cw).fit(Xtr, ytr)
    pr = m.predict(Xte)
    print(f"class_weight {str(cw):9}: recall {recall_score(yte, pr):.3f}  precision {precision_score(yte, pr):.3f}  average predicted probability {m.predict_proba(Xte)[:, 1].mean():.3f}")
print("true share of positives in the test set:", round(yte.mean(), 3))

class_weight None     : recall 0.075  precision 0.348  average predicted probability 0.055
class_weight balanced : recall 0.822  precision 0.187  average predicted probability 0.314
true share of positives in the test set: 0.054


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [6]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
cols = ["dist_km", "age_yrs"]
clf = make_pipeline(StandardScaler(), LogisticRegression()).fit(df[cols], df["sold_fast"])
print({c: round(float(v), 2) for c, v in zip(cols, clf[-1].coef_[0])})
new = pd.DataFrame({"dist_km": [3, 20], "age_yrs": [5, 25]})
print(clf.predict_proba(new)[:, 1].round(2))      # P(sells fast) for two new houses

{'dist_km': -1.04, 'age_yrs': -1.27}
[0.88 0.03]


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [7]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: The sigmoid

Write the sigmoid so it works on whole arrays.

In [8]:
import numpy as np
z = np.array([-2.0, 0.0, 1.5])
p = None  # ← your code here
check(p, [0.11920292202211755, 0.5, 0.8175744761936437], hint='σ(z) = 1 / (1 + e^(−z)); use np.exp.')

❌ Not yet. Hint: σ(z) = 1 / (1 + e^(−z)); use np.exp.


### Challenge 2: Log loss

Compute the average log loss for these predictions.

In [9]:
import numpy as np
y = np.array([1, 0, 1, 0])
p = np.array([0.9, 0.2, 0.6, 0.4])
loss = None  # ← your code here
check(loss, 0.3375388286260044, hint='−mean(y·ln p + (1 − y)·ln(1 − p)).')

❌ Not yet. Hint: −mean(y·ln p + (1 − y)·ln(1 − p)).


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. The sigmoid** — the missing line:

```python
p = 1 / (1 + np.exp(-z))
```

**2. Log loss** — the missing line:

```python
loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
```